In [38]:
import pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import json
import csv
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .enableHiveSupport()
    .appName("cu2")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .getOrCreate()
)
sql = spark.sql


In [39]:
df = pd.read_csv(
    "/data/scratch/ldedieu/CU2/encoder_baseline/MISTRAL/test_format2.csv",
    #quoting=csv.QUOTE_NONE,
    on_bad_lines='skip'
)

In [40]:
len(df)

2249

In [41]:
df = df.dropna()
len(df)

1990

In [42]:
df.columns

Index(['index', 'clinical_note', 'procedure_json', 'annot_diagnostics_v2'], dtype='str')

In [43]:
df = df[["clinical_note", "annot_diagnostics_v2"]]

In [44]:
import pandas as pd
import ast

def extract_all_diagnostics(cell_value):
    # Initialisation des résultats par défaut
    res = {'dp': None, 'dr': None, 'das': []}
    
    if pd.isna(cell_value):
        return pd.Series(res)
    
    if isinstance(cell_value, str):
        try:
            cell_value = ast.literal_eval(cell_value)
        except (ValueError, SyntaxError):
            return pd.Series(res)
            
    if isinstance(cell_value, dict):
        for key, details in cell_value.items():
            if isinstance(details, dict):
                pos = details.get('position')
                if pos == 'DP':
                    res['dp'] = key
                elif pos == 'DR':
                    res['dr'] = key
                elif pos == 'DAS':
                    res['das'].append(key)
                    
    return pd.Series(res)

# Application sur la colonne pour créer les 3 nouvelles colonnes en une seule passe
df[['dp', 'dr', 'das']] = df['annot_diagnostics_v2'].apply(extract_all_diagnostics)
df['das'] = df['das'].apply(lambda x: list(set(x)) if isinstance(x, list) else x)
df = df.drop("annot_diagnostics_v2", axis=1)

In [45]:
df

,clinical_note,dp,dr,das
0,SERVICE DE CHIRURGIE VASCULAIRE\nHÔPITAL MICHA...,I839,None,"[J961+, K743, E10, I10]"
1,Service de Chirurgie Viscérale – Hôpital Roger...,K409,None,"[E890, Z991, M1099, M4712, F1024, R64, G473, I10]"
2,SERVICE D'ORTHOPÉDIE ET TRAUMATOLOGIE\nHÔPITAL...,S934,None,"[Z993, S5250, E785, C787, C793, C50, W00, I10]"
4,SERVICE DE NEURO-CHIRURGIE – HÔPITAL MICHALLON...,S0650,None,[D638]
6,"Patient : Alain Bensaad, 50 ans\nDate d’interv...",C771,None,"[Z538, C778, D611, C773, F10240, C786, J961+, ..."
...,...,...,...,...
2243,Exérèse de tumeur du foramen magnum par cranio...,C793,None,"[Z974, C783, R42, Z8670, C788, C794, Z298]"
2245,SERVICE DE CARDIOLOGIE – HÔPITAL PITIÉ-SALPÊTR...,Z450,None,"[I340, B962]"
2246,Hôpital Michallon - CHU Grenoble Alpes\nServic...,R101,None,"[C786, C189, I342, F209]"
2247,Hôpital Bichat-Claude Bernard - Assistance Pub...,F239,None,"[M5429, R410, F220, R440, F142, F2099]"


In [46]:
df['mdp'] = 'Z769'

mask = df['dr'].notna()

df.loc[mask, 'mdp'] = df.loc[mask, 'dp']
df.loc[mask, 'dp'] = df.loc[mask, 'dr']

df.drop(columns=['dr'], inplace=True)

In [47]:
df

,clinical_note,dp,das,mdp
0,SERVICE DE CHIRURGIE VASCULAIRE\nHÔPITAL MICHA...,I839,"[J961+, K743, E10, I10]",Z769
1,Service de Chirurgie Viscérale – Hôpital Roger...,K409,"[E890, Z991, M1099, M4712, F1024, R64, G473, I10]",Z769
2,SERVICE D'ORTHOPÉDIE ET TRAUMATOLOGIE\nHÔPITAL...,S934,"[Z993, S5250, E785, C787, C793, C50, W00, I10]",Z769
4,SERVICE DE NEURO-CHIRURGIE – HÔPITAL MICHALLON...,S0650,[D638],Z769
6,"Patient : Alain Bensaad, 50 ans\nDate d’interv...",C771,"[Z538, C778, D611, C773, F10240, C786, J961+, ...",Z769
...,...,...,...,...
2243,Exérèse de tumeur du foramen magnum par cranio...,C793,"[Z974, C783, R42, Z8670, C788, C794, Z298]",Z769
2245,SERVICE DE CARDIOLOGIE – HÔPITAL PITIÉ-SALPÊTR...,Z450,"[I340, B962]",Z769
2246,Hôpital Michallon - CHU Grenoble Alpes\nServic...,R101,"[C786, C189, I342, F209]",Z769
2247,Hôpital Bichat-Claude Bernard - Assistance Pub...,F239,"[M5429, R410, F220, R440, F142, F2099]",Z769


In [48]:
df = df[df['dp'].apply(lambda x: isinstance(x, str))]
len(df)

1981

In [49]:
codes = pd.read_csv(
    "../data/Referentiel_CIM-10-20250108.csv", sep=";", header=1, dtype=str
)

codes = codes[["code", "libellé long", "code MCO/HAD"]].rename(
    columns={"libellé long": "definition", "code MCO/HAD": "mco_had"}
)
codes["code"] = codes["code"].astype(str).str.strip()
codes["mco_had"] = pd.to_numeric(codes["mco_had"], errors="coerce")

dp = codes[codes["mco_had"].isin([0])].drop("mco_had", axis=1)
das = codes[codes["mco_had"].isin([0,1,2])].drop("mco_had", axis=1)

dp=set(dp["code"].to_list())
das=set(das["code"].to_list())
print(len(dp))
print(len(das))

13358
40333


In [50]:
df = df[df['dp'].isin(dp)]
print(len(df))
df = df[df['mdp'].isin(dp)]
print(len(df))
df['das'] = df['das'].apply(
    lambda x: [code for code in x if code in das] if isinstance(x, list) else x
)
print(len(df))

1823
1823
1823


In [51]:
df_train = pd.read_csv(
    "/data/scratch/ldedieu/CU2/encoder_baseline/MISTRAL/train_format2.csv",
    #quoting=csv.QUOTE_NONE,
    on_bad_lines='skip'
)

In [52]:
df_train

,index,clinical_note,procedure_json,annot_diagnostics_v2
0,8499,Nouvel Hôpital Civil - Hôpitaux Universitaires...,"[""HGLE001""]","{'E86': {'position': 'DP', 'proofs': ['déshydr..."
1,1041,Hôpital Nord - Assistance Publique – Hôpitaux ...,"[""MJFA003""]","{'L030': {'position': 'DP', 'proofs': ['phlegm..."
2,101,"M. Sasa Fauche, 60 ans, est admis en hospitali...","[""ZZHJ021""]","{'C61': {'position': 'DP', 'proofs': ['adénoca..."
3,8681,Hôpital Saint-Louis - Assistance Publique Hôpi...,"[""DDQH009""]","{'I352': {'position': 'DP', 'proofs': ['sténos..."
4,3657,Service de Pédiatrie Générale – Hôpital Pelleg...,"[""AMQP011""]","{'G478': {'position': 'DP', 'proofs': ['troubl..."
...,...,...,...,...
10494,7616,SERVICE DE RADIOTHÉRAPIE\nHôpital Larrey - Cen...,"[""ZZNL045""]","{'C797': {'position': 'DP', 'proofs': ['métast..."
10495,2975,"Madame Nafi Picot, 52 ans, est hospitalisée le...","[""ZZHJ013""]","{'C187': {'position': 'DP', 'proofs': ['carcin..."
10496,6235,HÔPITAL DE LA CROIX-ROUSSE - HOSPICES CIVILS D...,"[""GLLD008""]","{'T402': {'position': 'DP', 'proofs': ['intoxi..."
10497,7656,Hôpital Bichat-Claude Bernard - Assistance Pub...,"[""BZQK001""]","{'S05': {'position': 'DP', 'proofs': ['traumat..."


In [57]:
n=1000
df['existe_dans_df2'] = df['clinical_note'].str[:n].isin(df_train['clinical_note'].str[:n])

In [58]:
df['existe_dans_df2'].value_counts()

existe_dans_df2
False    1822
True        1
Name: count, dtype: int64

In [59]:
df_spark = spark.createDataFrame(df) \
    .withColumnRenamed("clinical_note", "note_text") \
    .withColumnRenamed("icd_primary_code", "dp")

In [60]:
df_spark.show(2,vertical=True)

-RECORD 0-------------------------------
 note_text       | SERVICE DE CHIRUR... 
 dp              | I839                 
 das             | [K743, I10]          
 mdp             | Z769                 
 existe_dans_df2 | false                
-RECORD 1-------------------------------
 note_text       | Service de Chirur... 
 dp              | K409                 
 das             | [E890, Z991, M109... 
 mdp             | Z769                 
 existe_dans_df2 | false                
only showing top 2 rows



In [61]:
df = df_spark

In [62]:
df = df.filter(
    (F.col("dp").isNotNull()) & (F.col("dp") != "")
)

In [63]:
labels_dp = df.select("dp").distinct().rdd.flatMap(lambda x: x).collect()
labels_mdp = df.select("mdp").distinct().rdd.flatMap(lambda x: x).collect()
labels_das = df.select(F.explode("das")).distinct().rdd.flatMap(lambda x: x).collect()
print(len(labels_dp))
print(len(labels_mdp))
print(len(labels_das))

[Stage 7:====================================================>(1981 + 5) / 2000]

854
1
2167


In [64]:
label_counts_dp = (
    df.groupBy("dp")
      .agg(F.count("*").alias("count"))
      .orderBy(F.desc("count"))
)

label_counts_mdp = (
    df.groupBy("mdp")
      .agg(F.count("*").alias("count"))
      .orderBy(F.desc("count"))
)

label_counts_das = (
    df.select(F.explode("das").alias("code_das"))
      .groupBy("code_das")
      .count()
      .orderBy(F.desc("count"))
)
print(label_counts_dp.count())
print(label_counts_mdp.count())
print(label_counts_das.count())
#label_counts_dp.show()

854


1


[Stage 23:===================================================>(1982 + 5) / 2000]

2167


In [65]:
label_counts_das.show(10)

[Stage 27:=================================================>  (1913 + 5) / 2000]

+--------+-----+
|code_das|count|
+--------+-----+
|     I10|  637|
|    C787|  304|
|    C795|  266|
|    C780|  249|
|    C786|  169|
|    C793|  126|
|    C771|  126|
|    C772|  124|
|    Z515|  102|
|    C797|   90|
+--------+-----+
only showing top 10 rows



In [72]:
from pyspark.sql import functions as F

def save_df_to_parquet(df, filename, num_partitions):
    df = df.withColumn("note_id", F.monotonically_increasing_id())

    df_to_save = df.select(
        "note_id",
        F.col("note_text"),
        F.col("dp"),
        F.col("das"),
        F.col("mdp"),
    )

    df_to_save.repartition(num_partitions).write.mode("overwrite").parquet(filename)


save_df_to_parquet(df, "CU2/encoder_baseline/MISTRAL/test_v1_das", num_partitions=2)


In [73]:
import subprocess
user="ldedieu"

subprocess.run(
    #["mkdir", "-p", f"/data/scratch/{user}/ia_codage/DP_DR/{strat}_{n_class_dp}class_v2"],
    ["mkdir", "-p", f"/data/scratch/{user}/CU2/encoder_baseline/MISTRAL/test_v1_das"],
    check=True
)

subprocess.run(
    [
        "hdfs", "dfs", "-copyToLocal", "-f",
        #f"ia_codage/dataset_800k/{strat}_{n_class_dp}class_v2/*",
        #f"/data/scratch/{user}/ia_codage/DP_DR/{strat}_{n_class_dp}class_v2/"
        f"CU2/encoder_baseline/MISTRAL/test_v1_das/*",
        f"/data/scratch/{user}/CU2/encoder_baseline/MISTRAL/test_v1_das/"
    ],
    check=True
)

CompletedProcess(args=['hdfs', 'dfs', '-copyToLocal', '-f', 'CU2/encoder_baseline/MISTRAL/test_v1_das/*', '/data/scratch/ldedieu/CU2/encoder_baseline/MISTRAL/test_v1_das/'], returncode=0)

In [74]:
df.show(2)

+--------------------+----+--------------------+----+---------------+
|           note_text|  dp|                 das| mdp|existe_dans_df2|
+--------------------+----+--------------------+----+---------------+
|SERVICE DE CHIRUR...|I839|         [K743, I10]|Z769|          false|
|Service de Chirur...|K409|[E890, Z991, M109...|Z769|          false|
+--------------------+----+--------------------+----+---------------+
only showing top 2 rows

